In [21]:
from langgraph.graph import StateGraph, START, END
from pydantic import BaseModel
from typing import Literal

In [22]:
class TicketState(BaseModel):
    issue : str = ""
    category : str = ""
    response: str = ""

In [23]:
def classify(state:TicketState) -> TicketState:
    issue = state.issue.lower()
    if "payment" in issue or "amount" in issue:
        state.category = "billing"
    elif "refund" in issue or "cancel" in  issue:
        state.category = "refund"
    else:
        state.category = "status"


    return state

In [24]:
def handle_billing(state:TicketState) -> TicketState:
    print('Billing Related Issue')
    state.response = f"Issue Resolved for {state.category}"
    return state



def handle_refund(state:TicketState) -> TicketState:
    print('Refund Related Issue')
    state.response = f"Issue Resolved for {state.category}"
    return state


def handle_status(state:TicketState) -> TicketState:
    print('Status Related Issue')
    state.response = f"Issue Resolved for {state.category}"
    return state

In [25]:
def router_logic(state:TicketState) -> Literal["billing","refund","status"]:
    return state.category

In [26]:
graph = StateGraph(TicketState)
graph.add_node("classify", classify)
graph.add_node("handle_billing",handle_billing)
graph.add_node("handle_refund",handle_refund)
graph.add_node("handle_status",handle_status)

graph.add_edge(START, "classify")
graph.add_conditional_edges("classify", router_logic),
{"billing":"handle_billing", "status":"handle_status", "refund":"handle_refund"}


graph.add_edge("handle_billing", END)
graph.add_edge("handle_refund", END)
graph.add_edge("handle_status", END)


graph = StateGraph(TicketState)
graph.add_node("classify", classify)
graph.add_node("handle_billing", handle_billing)
graph.add_node("handle_refund", handle_refund)
graph.add_node("handle_status", handle_status)

graph.add_edge(START, "classify")
graph.add_conditional_edges(
    "classify",
    router_logic,
    {
        "billing": "handle_billing",
        "status": "handle_status",
        "refund": "handle_refund",
    },
)

graph.add_edge("handle_billing", END)
graph.add_edge("handle_refund", END)
graph.add_edge("handle_status", END)

final_graph = graph.compile()

In [27]:
res = final_graph.invoke({"issue": "I am waiting for my refund"})

Refund Related Issue


In [28]:
res

{'issue': 'I am waiting for my refund',
 'category': 'refund',
 'response': 'Issue Resolved for refund'}

### 5.Memory - CheckPointer


In [ ]:
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage
from langchain_groq import ChatGroq
from pydantic import BaseModel
from typing import Annotated

llm = ChatGroq(model="openai/gpt-oss-20b")

class ChatState(BaseModel):
    messages: Annotated[list, add_messages]


def chat_bot(state:ChatState):
    state.messages = [llm.invoke(state.messages)]
    return state


graph = StateGraph(ChatState)

graph.add_node("chat_bot", chat_bot)
graph.add_node(START,"chat_bot")
graph.add_node("chat_bot", END)

memory = InMemorySaver()
final = graph.compile(checkpointer=memory)

AttributeError: 'StateGraph' object has no attribute 'add'

In [ ]:
res = final.invoke(
    {"messages": [HumanMessage(content="What is his age ?") ]
    },
                   {"configurable": {"thread_id":"id_1"}}
    )
ans = res["messages"][1].content
print(res)
